# HOW TO GLIMT

In [1]:
import load_secrets, os
load_secrets.load_secrets()

## JSX Requests

The JSX requests using curl have this format:

In [2]:
import os, json, requests

def jsx_request(jsxRequest):
    url = "https://glimt.nu/glimt-jsx/jsx.json?lang=en"
    
    headers = {
        "Content-Type": "application/json;charset=utf-8"
    }
    
    cookies = {
        "lumAuth": os.getenv("GLIMT_API_KEY")
    }
       
    response = requests.post(url, headers=headers, cookies=cookies, data=jsxRequest)
    
    print(response.status_code)

    return json.loads(response.text)

**Response text:** the text returned by a JSX request is itself always wrapped inside a JSON array. Therefore, below, when we say that a request returns value X, it really means that it returns [ X ] . 

If the response  text does not start by '[', i.e. is not a JSON Array, it indicates an error which is described more or less opaquely in the reply.

## Dates expressed as offsets from Unix day 0

In [99]:
from datetime import datetime, timedelta

def days_from_0_to_datetime(day):
    seconds_per_day =  86400
    days_in_s = (day-1) * seconds_per_day
    dt = datetime.fromtimestamp(days_in_s)-timedelta(hours=19)
    idt = int(str(dt)[0:10].replace('-',''))
    return idt

if __name__=="__main__":
    # should map 20104 to Jan. 14, 2025 and 20454 to Dec. 30, 2025. 
    print([(x, days_from_0_to_datetime(x)) for x in [20104, 20454]])

[(20104, 20250114), (20454, 20251230)]


## Query active IFPs

To request the list of active IFPs, replace jsxRequest by:

In [3]:
def list_active_ifps():
    L = jsx_request("""[["ifps", "queryIFPs", {query: {states: ["active"]}, fmt: {}}]]""")
    return L[0]

It will return a JSON array of all active iFPs and their detailed properties, including:
* **symbol**
* **title**
* **details**: Information beyond the title of the IFP, such as what sources may be used to resolve the IFP, and/or some background information that might be useful to forecasters, &c.
* **bins**: An array of the proposed resolution outcomes

In [102]:
ifps = list_active_ifps()

200


In [104]:
ifps_back = ifps.copy()

In [103]:
len(ifps)

19

## Convert dates in IFPs

In [105]:
for ifp in ifps:
    for key in ifp['dates']:
        value = ifp['dates'][key]
        ifp['dates'][key] = days_from_0_to_datetime(value)

In [106]:
ifps[10]['dates'][key]

20251230

## Let's start saving things

In [107]:
os.makedirs('glimt/ifp', exist_ok=True)

In [108]:
from tqdm import tqdm
import json

In [110]:
for ifp in tqdm(ifps):
    id = ifp['id']
    fn = f'glimt/ifp/{id}.json'
    with open(fn, 'w') as f:
        json.dump(ifp, f, indent=4)

100%|█████████████████████████████████████████| 19/19 [00:00<00:00, 8852.67it/s]


## Viewed as a dataframe

In [116]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df = pd.DataFrame([(ifp['id'], ifp['symbol'], ifp['dates']['startDay'], ifp['dates']['endDay'], 
                    ifp['props']['shortTitle']) for ifp in ifps], columns = ['id', 'challenge', 'startDay', 'endDay', 'title'])

In [119]:
df.sort_values(by=['challenge', 'endDay', 'title'])

,id,challenge,startDay,endDay,title
5,471,Eco_NORDSTREAM_Jan_25,20250114,20251230,Will Nord Stream be back in 2025?
8,475,Eco_RUSSIA_FUND_Jan_25,20250116,20250930,Russia to run out of cash by October?
2,468,Economics_RESERVES_Jan_25,20250114,20251230,When will Russia get back its frozen foreign exchange reserves?
0,458,Georgiainvasion,20250114,20251230,Will Russia invade Georgia in 2025?
18,502,Light_BelarusProtest,20250515,20251230,Protests in Belarus?
14,493,Light_ELONRICH_25,20250326,20251230,Will Elon Musk remain the richest in 2025?
17,501,Light_GDPG20Q2_May25,20250513,20250913,Economic growth in the G20-countries during Q2?
13,492,Light_TrumpNobel_25,20250326,20251009,Trump to win the 2025 Nobel Peace Prize?
15,498,Light_US_SouthKorea_Apr_25,20250428,20251230,American troops out of South Korea?
16,499,Pol_CHECHNYA_May_25,20250504,20251130,Ramzan Kadyrov still Head of Chechnya on Dec. 1?


## News for each question

In [120]:
from call_asknews import call_asknews

In [128]:
os.makedirs('glimt/news', exist_ok=True)

In [135]:
news = {}
for ifp in ifps:
    id, title, details = ifp['id'], ifp['props']['title'], ifp['props']['details']
    fn = f'glimt/news/{id}.txt'
    if os.path.exists(fn):
        with open(fn, 'r') as f:
            news[id] = f.read()
        continue
    prompt = f"""{title}\n{details}"""
    news[id] = call_asknews(prompt, True)
    with open(fn, 'w') as f:
        f.write(news[id])
    print('saved', fn)

saved glimt/news/463.txt
saved glimt/news/468.txt
saved glimt/news/469.txt
saved glimt/news/470.txt
saved glimt/news/471.txt
saved glimt/news/472.txt
saved glimt/news/473.txt
saved glimt/news/475.txt
saved glimt/news/476.txt
saved glimt/news/485.txt
saved glimt/news/489.txt
saved glimt/news/490.txt
saved glimt/news/492.txt
saved glimt/news/493.txt
saved glimt/news/498.txt
saved glimt/news/499.txt
saved glimt/news/501.txt
saved glimt/news/502.txt


In [131]:
len(ifps)

19

In [133]:
len(news)

1

## Submit forecast to IFP

To submit a forecast, replace jsxRequest by:

In [6]:
def submit_forecast(symbol, rationale, binProbas):
    return jsx_request(f"""[["ifps","submitAIFcst",{"ifpRef": "{symbol}","data": {"probas": {binProbas} },"reasoning": {rationale}}]]""")

where you should replace 
* **symbol** with the symbol of the IFP for which you are submitting a forecast bin
* **binProbas** with a JSON array of probabilities adding to 1.0, and such that each one corresponds to the IFP's bin (i.e. outcome) at the same index. For example, an IFP with 4 outcomes could accept [0.2, 0.6, 0, 0.2] where 0.6 is the probability you assign to the second outcome.
* **rationale** with your reasoning

It will return a JSON object containing the full description of your submitted forecast, including the forecast ID.